# 01 · Exploratory Data Analysis: Crop Recommendation Dataset

**Dataset:** `ml-service/data/Crop_recommendation.csv` (Kaggle, 2,200 rows, 22 crops).

**Features:** N, P, K (soil nutrients, kg/ha), temperature (°C), humidity (%), ph, rainfall (mm). **Label:** `label`.

Goals: check data quality, understand feature distributions, see how separable the crops are, and look for correlated features.

In [1]:
# Locate the project root (works when launched from notebooks/ or the repo root)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml-service").is_dir())
ML = ROOT / "ml-service"
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)
sys.path.insert(0, str(ML))  # lets us import the shared training/ package
print("Project root:", ROOT)

Project root: D:\AI powered\agri-dss


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from training import charts  # shared, validated report palette + styling

df = pd.read_csv(ML / "data" / "Crop_recommendation.csv")
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
print(df.shape)
df.head()

(2200, 8)


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


## 1. Data quality: types, missing values, duplicates

In [3]:
df.info()
print("\nMissing values per column:\n", df.isna().sum().to_string())
print("\nDuplicate rows:", df.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   ph           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   str    
dtypes: float64(4), int64(3), str(1)
memory usage: 137.6 KB

Missing values per column:
 N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0

Duplicate rows: 0


## 2. Summary statistics

In [4]:
df[FEATURES].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
N,2200.0,50.55,36.92,0.00,21.00,37.00,84.25,140.00
P,2200.0,53.36,32.99,5.00,28.00,51.00,68.00,145.00
K,2200.0,48.15,50.65,5.00,20.00,32.00,49.00,205.00
temperature,2200.0,25.62,5.06,8.83,22.77,25.60,28.56,43.68
humidity,2200.0,71.48,22.26,14.26,60.26,80.47,89.95,99.98
ph,2200.0,6.47,0.77,3.50,5.97,6.43,6.92,9.94
rainfall,2200.0,103.46,54.96,20.21,64.55,94.87,124.27,298.56


## 3. Class balance
Each of the 22 crops has exactly 100 samples, so the dataset is perfectly balanced. Accuracy is therefore a fair headline metric, and we still report macro F1.

In [5]:
counts = df["label"].value_counts().sort_index()
print("crops:", counts.size, "| min:", counts.min(), "| max:", counts.max())
counts.to_frame("samples").T

crops: 22 | min: 100 | max: 100


label,apple,banana,blackgram,chickpea,coconut,coffee,cotton,grapes,jute,kidneybeans,...,mango,mothbeans,mungbean,muskmelon,orange,papaya,pigeonpeas,pomegranate,rice,watermelon
samples,100,100,100,100,100,100,100,100,100,100,...,100,100,100,100,100,100,100,100,100,100


## 4. Feature distributions

In [6]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, f in zip(axes.flat, FEATURES):
    ax.hist(df[f], bins=30, color=charts.SERIES[0], edgecolor=charts.SURFACE, linewidth=1)
    ax.set_title(f, loc="left")
    ax.grid(axis="y", color=charts.GRID, linewidth=0.8); ax.set_axisbelow(True)
axes.flat[-1].axis("off")
fig.suptitle("Feature distributions (all 2,200 samples)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
fig.savefig(REPORTS / "eda_distributions.png", dpi=150)
plt.show()

C:\Users\bhara\AppData\Local\Temp\ipykernel_28104\1476074038.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. How each feature separates the crops
Boxplots per crop show which features are discriminative. For example, K is very high for apple and grapes, rainfall is high for rice and coconut, and humidity is low for chickpea and kidneybeans.

In [7]:
fig, axes = plt.subplots(len(FEATURES), 1, figsize=(14, 3.0 * len(FEATURES)), sharex=True)
order = sorted(df["label"].unique())
for ax, f in zip(axes, FEATURES):
    sns.boxplot(data=df, x="label", y=f, order=order, ax=ax, color=charts.SERIES[0],
                linewidth=0.8, fliersize=2, boxprops={"alpha": 0.85})
    ax.set_xlabel(""); ax.grid(axis="y", color=charts.GRID, linewidth=0.8); ax.set_axisbelow(True)
axes[-1].tick_params(axis="x", rotation=60)
fig.suptitle("Feature ranges by crop", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
fig.savefig(REPORTS / "eda_boxplots_by_crop.png", dpi=120)
plt.show()

C:\Users\bhara\AppData\Local\Temp\ipykernel_28104\657092371.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Correlation between features
Diverging palette: orange = negative, neutral grey = 0, blue = positive.

In [8]:
diverging = LinearSegmentedColormap.from_list("div", ["#eb6834", "#f3f2ef", "#2a78d6"])
corr = df[FEATURES].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap=diverging, vmin=-1, vmax=1, square=True,
            linewidths=2, linecolor=charts.SURFACE, cbar_kws={"shrink": 0.8, "label": "Pearson r"}, ax=ax)
ax.set_title("Feature correlation matrix", loc="left")
fig.tight_layout()
fig.savefig(REPORTS / "eda_correlation.png", dpi=150)
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().sort_values(key=abs, ascending=False)
print("Strongest correlations:\n", pairs.head(3).round(3).to_string())

Strongest correlations:
 P            K           0.736
N            P          -0.231
temperature  humidity    0.205


C:\Users\bhara\AppData\Local\Temp\ipykernel_28104\3765840036.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Per-crop ideal ranges
The per-crop mean ± std of each feature is saved to `knowledge/crop_profiles.json` by notebook 02. It drives the **weather score** (live temperature, humidity and rainfall vs the crop's ideal range) and the **fertilizer rules** (ideal NPK = per-crop mean).

In [9]:
df.groupby("label")[FEATURES].mean().round(1)

,N,P,K,temperature,humidity,ph,rainfall
label,,,,,,,
apple,20.8,134.2,199.9,22.6,92.3,5.9,112.7
banana,100.2,82.0,50.0,27.4,80.4,6.0,104.6
blackgram,40.0,67.5,19.2,30.0,65.1,7.1,67.9
chickpea,40.1,67.8,79.9,18.9,16.9,7.3,80.1
coconut,22.0,16.9,30.6,27.4,94.8,6.0,175.7
coffee,101.2,28.7,29.9,25.5,58.9,6.8,158.1
cotton,117.8,46.2,19.6,24.0,79.8,6.9,80.4
grapes,23.2,132.5,200.1,23.8,81.9,6.0,69.6
jute,78.4,46.9,40.0,25.0,79.6,6.7,174.8


## Findings
- **Clean data:** no missing values and no duplicate rows; 22 balanced classes of 100 samples each.
- **P and K are strongly correlated** (r ≈ 0.74), driven by apple and grapes, which need very high P and K. Tree models handle this well.
- Features are on very different scales (pH 3.5–9.9 vs rainfall 20–300 mm), so **StandardScaler** is needed for the linear and SVM baselines.
- Several crops are separable on one or two features (rainfall, humidity, K), so high accuracy is expected. The closest crops are the pulses (blackgram, mungbean, mothbeans, lentil), which have overlapping NPK and climate ranges. In notebook 02, the only two test-set errors of the stacked model are blackgram→maize and lentil→mothbeans.